# 06 SOKE Gemma3 LoRA Rank Sweep

One-epoch `lm_pretrain` sweep for LoRA rank/alpha pairs. The notebook requests `attn_implementation="flash_attention_2"` by default, logs every optimizer step, validates after the epoch, and plots train-step loss plus validation loss in one figure.

In [ ]:
# Run configuration: edit this cell first in Colab.
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

IN_COLAB = Path('/content').exists()
DEFAULT_DRIVE_BUNDLE_ROOT = Path('/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm')
DEFAULT_LOCAL_BUNDLE_ROOT = Path('/content/04_soke_gemma3_270m_lora_lm') if IN_COLAB else Path.cwd()

DRIVE_BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_SWEEP_DRIVE_BUNDLE_ROOT', str(DEFAULT_DRIVE_BUNDLE_ROOT)))
LOCAL_BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_SWEEP_LOCAL_BUNDLE_ROOT', str(DEFAULT_LOCAL_BUNDLE_ROOT)))
BASE_MODEL = os.environ.get('SOKE_GEMMA_SWEEP_BASE_MODEL', 'google/gemma-3-270m')

# A100 and RTX PRO 6000 Blackwell should use FlashAttention-2. H100/Hopper can also use FA3, but that is not the target here.
ATTN_IMPLEMENTATION = os.environ.get('SOKE_GEMMA_SWEEP_ATTN_IMPLEMENTATION', 'flash_attention_2')
SDPA_KERNEL = os.environ.get('SOKE_GEMMA_SWEEP_SDPA_KERNEL', 'auto')  # only used when ATTN_IMPLEMENTATION='sdpa'
INSTALL_FLASH_ATTN_IF_MISSING = os.environ.get('SOKE_GEMMA_SWEEP_INSTALL_FLASH_ATTN', '1') == '1'

# Sweep ranks and alpha/r ratios. Current baseline was alpha/r = 2.
LORA_RANKS = [1024, 512, 256, 128, 64]
LORA_ALPHA_RATIOS = [0.5, 1.0, 2.0]
LORA_SWEEP = [(rank, int(rank * ratio), ratio) for rank in LORA_RANKS for ratio in LORA_ALPHA_RATIOS]

EPOCHS = int(os.environ.get('SOKE_GEMMA_SWEEP_EPOCHS', '1'))
HARDWARE_BATCH_SIZE = int(os.environ.get('SOKE_GEMMA_SWEEP_BATCH_SIZE', '16'))
VIRTUAL_BATCH_SIZE = int(os.environ.get('SOKE_GEMMA_SWEEP_VIRTUAL_BATCH_SIZE', '128'))
GRAD_ACCUM = max(1, VIRTUAL_BATCH_SIZE // max(1, HARDWARE_BATCH_SIZE))
NUM_WORKERS = int(os.environ.get('SOKE_GEMMA_SWEEP_NUM_WORKERS', '8'))
PIN_MEMORY = int(os.environ.get('SOKE_GEMMA_SWEEP_PIN_MEMORY', '1'))
PERSISTENT_WORKERS = int(os.environ.get('SOKE_GEMMA_SWEEP_PERSISTENT_WORKERS', '1'))
PREFETCH_FACTOR = int(os.environ.get('SOKE_GEMMA_SWEEP_PREFETCH_FACTOR', '4'))
TF32 = int(os.environ.get('SOKE_GEMMA_SWEEP_TF32', '1'))
TORCH_COMPILE = int(os.environ.get('SOKE_GEMMA_SWEEP_TORCH_COMPILE', '0'))
TORCH_COMPILE_MODE = os.environ.get('SOKE_GEMMA_SWEEP_TORCH_COMPILE_MODE', 'reduce-overhead')
MAX_SEQ_LEN = int(os.environ.get('SOKE_GEMMA_SWEEP_MAX_SEQ_LEN', '1024'))
MAX_TRAIN_LOGICAL_ROWS = int(os.environ.get('SOKE_GEMMA_SWEEP_MAX_TRAIN_LOGICAL_ROWS', '0'))
MAX_VAL_LOGICAL_ROWS = int(os.environ.get('SOKE_GEMMA_SWEEP_MAX_VAL_LOGICAL_ROWS', '2048'))
EVAL_MAX_BATCHES = int(os.environ.get('SOKE_GEMMA_SWEEP_EVAL_MAX_BATCHES', '0'))
LR = float(os.environ.get('SOKE_GEMMA_SWEEP_LR', '2e-4'))
MOTION_TOKEN_LR = float(os.environ.get('SOKE_GEMMA_SWEEP_MOTION_TOKEN_LR', '2e-5'))
ETA_MIN = float(os.environ.get('SOKE_GEMMA_SWEEP_ETA_MIN', '1e-6'))
LORA_DROPOUT = float(os.environ.get('SOKE_GEMMA_SWEEP_LORA_DROPOUT', '0.05'))
DTYPE = os.environ.get('SOKE_GEMMA_SWEEP_DTYPE', 'bf16')
TRAIN_MOTION_TOKEN_EMBEDDINGS = int(os.environ.get('SOKE_GEMMA_SWEEP_TRAIN_MOTION_TOKEN_EMBEDDINGS', '1'))
RANDOM_DROP = int(os.environ.get('SOKE_GEMMA_SWEEP_RANDOM_DROP', '1'))
SEED = int(os.environ.get('SOKE_GEMMA_SWEEP_SEED', '42'))

RUN_TRAINING = os.environ.get('SOKE_GEMMA_SWEEP_RUN_TRAINING', '1') == '1'
FORCE_RERUN = os.environ.get('SOKE_GEMMA_SWEEP_FORCE_RERUN', '0') == '1'
CONTINUE_ON_FAILURE = os.environ.get('SOKE_GEMMA_SWEEP_CONTINUE_ON_FAILURE', '1') == '1'
SYNC_TO_DRIVE = os.environ.get('SOKE_GEMMA_SWEEP_SYNC_TO_DRIVE', '1') != '0'
DRIVE_SYNC_RETRIES = int(os.environ.get('SOKE_GEMMA_SWEEP_DRIVE_SYNC_RETRIES', '5'))
DRIVE_SYNC_SLEEP_SEC = float(os.environ.get('SOKE_GEMMA_SWEEP_DRIVE_SYNC_SLEEP_SEC', '20'))

ATTN_RUN_LABEL = ATTN_IMPLEMENTATION.replace('_', '') if ATTN_IMPLEMENTATION != 'sdpa' else f'sdpa_{SDPA_KERNEL}'
OUTPUT_ROOT = LOCAL_BUNDLE_ROOT / '06_lora_rank_alpha_sweep'
DRIVE_OUTPUT_ROOT = DRIVE_BUNDLE_ROOT / '06_lora_rank_alpha_sweep'

print(json.dumps({
    'LOCAL_BUNDLE_ROOT': str(LOCAL_BUNDLE_ROOT),
    'DRIVE_BUNDLE_ROOT': str(DRIVE_BUNDLE_ROOT),
    'BASE_MODEL': BASE_MODEL,
    'ATTN_IMPLEMENTATION': ATTN_IMPLEMENTATION,
    'SDPA_KERNEL': SDPA_KERNEL,
    'LORA_SWEEP': LORA_SWEEP,
    'INSTALL_FLASH_ATTN_IF_MISSING': INSTALL_FLASH_ATTN_IF_MISSING,
    'EPOCHS': EPOCHS,
    'HARDWARE_BATCH_SIZE': HARDWARE_BATCH_SIZE,
    'GRAD_ACCUM': GRAD_ACCUM,
    'VIRTUAL_BATCH_SIZE': HARDWARE_BATCH_SIZE * GRAD_ACCUM,
    'NUM_WORKERS': NUM_WORKERS,
    'PIN_MEMORY': PIN_MEMORY,
    'PERSISTENT_WORKERS': PERSISTENT_WORKERS,
    'PREFETCH_FACTOR': PREFETCH_FACTOR,
    'TF32': TF32,
    'TORCH_COMPILE': TORCH_COMPILE,
    'TORCH_COMPILE_MODE': TORCH_COMPILE_MODE,
    'MAX_TRAIN_LOGICAL_ROWS': MAX_TRAIN_LOGICAL_ROWS,
    'MAX_VAL_LOGICAL_ROWS': MAX_VAL_LOGICAL_ROWS,
    'LR': LR,
    'MOTION_TOKEN_LR': MOTION_TOKEN_LR,
    'RUN_TRAINING': RUN_TRAINING,
    'SYNC_TO_DRIVE': SYNC_TO_DRIVE,
}, indent=2))

In [ ]:
# Mount Drive if needed, then copy scripts, instructions, and prebuilt SOKE motion-code JSONL data to local Colab storage.
import time

def run(cmd, *, check=True, env=None, retries=1, sleep_sec=5):
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    last = None
    for attempt in range(1, int(retries) + 1):
        print(f'[{attempt}/{retries}] {printable}', flush=True)
        last = subprocess.run([str(x) for x in cmd], check=False, env=env)
        if last.returncode == 0:
            return last
        if attempt < int(retries):
            time.sleep(float(sleep_sec))
    if check:
        raise subprocess.CalledProcessError(last.returncode, [str(x) for x in cmd])
    return last

def mount_drive_if_needed():
    if not IN_COLAB:
        return
    if Path('/content/drive/MyDrive').exists():
        return
    from google.colab import drive
    drive.mount('/content/drive')

def resolve_drive_bundle_root():
    global DRIVE_BUNDLE_ROOT
    if not IN_COLAB:
        return DRIVE_BUNDLE_ROOT
    mount_drive_if_needed()
    candidates = []
    for candidate in [
        DRIVE_BUNDLE_ROOT,
        Path('/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm'),
        Path('/content/drive/My Drive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm'),
    ]:
        if candidate not in candidates:
            candidates.append(candidate)
    for candidate in candidates:
        if candidate.exists():
            DRIVE_BUNDLE_ROOT = candidate
            return candidate
    parent = Path('/content/drive/MyDrive/folder/COLAB/Tokenizer')
    if parent.exists():
        print('Available Tokenizer dirs:', sorted(p.name for p in parent.iterdir() if p.is_dir())[:50])
    raise FileNotFoundError('Missing Drive bundle. Tried: ' + ', '.join(str(x) for x in candidates))

LOCAL_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
drive_root = resolve_drive_bundle_root() if IN_COLAB else DRIVE_BUNDLE_ROOT

if drive_root.exists() and drive_root.resolve() != LOCAL_BUNDLE_ROOT.resolve():
    for name in ['scripts', 'instructions']:
        src = drive_root / name
        dst = LOCAL_BUNDLE_ROOT / name
        if not src.exists():
            raise FileNotFoundError(f'Missing Drive folder: {src}')
        dst.mkdir(parents=True, exist_ok=True)
        run(['rsync', '-a', '--delete', str(src) + '/', str(dst) + '/'], retries=3)

    code_src = drive_root / 'outputs' / 'soke_motion_codes'
    code_dst = LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'
    if not code_src.exists():
        raise FileNotFoundError(f'Missing prebuilt motion-code folder on Drive: {code_src}')
    code_dst.mkdir(parents=True, exist_ok=True)
    run(['rsync', '-a', '--partial', '--append-verify', str(code_src) + '/', str(code_dst) + '/'], retries=3, sleep_sec=10)
elif not (LOCAL_BUNDLE_ROOT / 'scripts').exists():
    raise FileNotFoundError(f'Missing local bundle scripts and Drive bundle was not available: {LOCAL_BUNDLE_ROOT}')

code_root = LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'
for name in ['train_soke_motion_codes.jsonl', 'val_soke_motion_codes.jsonl', 'part_codecs.json']:
    path = code_root / name
    if not path.exists():
        raise FileNotFoundError(f'Missing required motion-code artifact: {path}')
    size = path.stat().st_size
    if size <= 0:
        raise RuntimeError(f'Empty required artifact, delete local copy and rerun this cell: {path}')
    print(path, size)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive bundle root:', drive_root)
print('Output root:', OUTPUT_ROOT)

In [ ]:
# Verify the selected fast attention backend before launching the sweep.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(f'{ATTN_IMPLEMENTATION} requires a CUDA GPU runtime.')

if ATTN_IMPLEMENTATION == 'flash_attention_2':
    try:
        import flash_attn
    except Exception as exc:
        if not INSTALL_FLASH_ATTN_IF_MISSING:
            raise RuntimeError('flash-attn is missing. Install it with: pip install -U flash-attn --no-build-isolation') from exc
        run([sys.executable, '-m', 'pip', 'install', '-U', 'flash-attn', '--no-build-isolation'])
        import flash_attn
    from transformers.utils import is_flash_attn_2_available
    if not is_flash_attn_2_available():
        raise RuntimeError('Transformers reports FlashAttention-2 is unavailable after flash-attn import.')
    print({
        'torch': torch.__version__,
        'cuda_device': torch.cuda.get_device_name(0),
        'attn_implementation': ATTN_IMPLEMENTATION,
        'flash_attn_version': getattr(flash_attn, '__version__', 'unknown'),
        'flash_attn_2_available': is_flash_attn_2_available(),
    })
elif ATTN_IMPLEMENTATION == 'sdpa' and SDPA_KERNEL == 'flash':
    if not hasattr(torch.backends.cuda, 'enable_flash_sdp'):
        raise RuntimeError('This PyTorch build does not expose Flash SDPA backend controls.')
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(False)
    print({
        'torch': torch.__version__,
        'cuda_device': torch.cuda.get_device_name(0),
        'attn_implementation': ATTN_IMPLEMENTATION,
        'sdpa_kernel': SDPA_KERNEL,
        'flash_sdp_enabled': torch.backends.cuda.flash_sdp_enabled(),
        'mem_efficient_sdp_enabled': torch.backends.cuda.mem_efficient_sdp_enabled(),
        'math_sdp_enabled': torch.backends.cuda.math_sdp_enabled(),
    })
else:
    print({'torch': torch.__version__, 'cuda_device': torch.cuda.get_device_name(0), 'attn_implementation': ATTN_IMPLEMENTATION, 'sdpa_kernel': SDPA_KERNEL})

In [ ]:
# Run one lm_pretrain epoch for each LoRA rank/alpha pair.
sweep_status = []
train_script = LOCAL_BUNDLE_ROOT / 'scripts' / 'train_gemma_soke_lora.py'
if not train_script.exists():
    raise FileNotFoundError(train_script)

for sweep_idx, (rank, alpha, alpha_ratio) in enumerate(LORA_SWEEP):
    ratio_label = str(alpha_ratio).replace('.', 'p')
    run_name = f'lm_pretrain_r{rank}_a{alpha}_ar{ratio_label}_{ATTN_RUN_LABEL}_e{EPOCHS}'
    local_run_root = OUTPUT_ROOT / 'runs' / run_name
    drive_run_root = DRIVE_OUTPUT_ROOT / 'runs' / run_name
    if FORCE_RERUN and local_run_root.exists():
        shutil.rmtree(local_run_root)
    if (local_run_root / 'final_status.json').exists() and not FORCE_RERUN:
        print('Skipping completed run:', local_run_root)
        sweep_status.append({'run_name': run_name, 'rank': rank, 'alpha': alpha, 'alpha_ratio': alpha_ratio, 'returncode': 0, 'skipped': True})
        continue
    cmd = [
        sys.executable, train_script,
        '--code-root', code_root,
        '--instructions-root', LOCAL_BUNDLE_ROOT / 'instructions',
        '--run-root', local_run_root,
        '--base-model', BASE_MODEL,
        '--stage', 'lm_pretrain',
        '--epochs', EPOCHS,
        '--resume-training', '0',
        '--batch-size', HARDWARE_BATCH_SIZE,
        '--grad-accum', GRAD_ACCUM,
        '--num-workers', NUM_WORKERS,
        '--pin-memory', PIN_MEMORY,
        '--persistent-workers', PERSISTENT_WORKERS,
        '--prefetch-factor', PREFETCH_FACTOR,
        '--tf32', TF32,
        '--torch-compile', TORCH_COMPILE,
        '--torch-compile-mode', TORCH_COMPILE_MODE,
        '--max-seq-len', MAX_SEQ_LEN,
        '--gradient-checkpointing', '1',
        '--attn-implementation', ATTN_IMPLEMENTATION,
        '--sdpa-kernel', SDPA_KERNEL,
        '--max-train-logical-rows', MAX_TRAIN_LOGICAL_ROWS,
        '--max-val-logical-rows', MAX_VAL_LOGICAL_ROWS,
        '--eval-max-batches', EVAL_MAX_BATCHES,
        '--validation-every-epochs', '1',
        '--motion-val-enabled', '0',
        '--lr', LR,
        '--motion-token-lr', MOTION_TOKEN_LR,
        '--eta-min', ETA_MIN,
        '--dtype', DTYPE,
        '--lora-r', rank,
        '--lora-alpha', alpha,
        '--lora-dropout', LORA_DROPOUT,
        '--lora-target-modules', 'auto',
        '--train-motion-token-embeddings', TRAIN_MOTION_TOKEN_EMBEDDINGS,
        '--random-drop', RANDOM_DROP,
        '--seed', SEED + sweep_idx,
        '--log-every-steps', '1',
        '--save-every-epochs', '1',
        '--keep-local-epoch-saves', '1',
        '--sync-to-drive', int(SYNC_TO_DRIVE and DRIVE_BUNDLE_ROOT.exists()),
        '--drive-sync-dest', drive_run_root,
        '--drive-sync-retries', DRIVE_SYNC_RETRIES,
        '--drive-sync-sleep-sec', DRIVE_SYNC_SLEEP_SEC,
        '--drive-sync-delete', '1',
        '--keep-drive-epoch-saves', '1',
    ]
    env = os.environ.copy()
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    if not RUN_TRAINING:
        print('Training disabled. Would run:', ' '.join(shlex.quote(str(x)) for x in cmd))
        continue
    result = run(cmd, check=False, env=env)
    sweep_status.append({'run_name': run_name, 'rank': rank, 'alpha': alpha, 'alpha_ratio': alpha_ratio, 'returncode': result.returncode, 'skipped': False})
    if result.returncode != 0 and not CONTINUE_ON_FAILURE:
        raise subprocess.CalledProcessError(result.returncode, cmd)

status_path = OUTPUT_ROOT / 'sweep_status.json'
status_path.write_text(json.dumps(sweep_status, indent=2), encoding='utf-8')
sweep_status

In [ ]:
# Aggregate optimizer-step train losses and epoch validation losses.
import math
import matplotlib.pyplot as plt
import pandas as pd

def read_jsonl(path):
    rows = []
    if not Path(path).exists():
        return rows
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

step_rows = []
summary_rows = []
for rank, alpha, alpha_ratio in LORA_SWEEP:
    ratio_label = str(alpha_ratio).replace('.', 'p')
    run_name = f'lm_pretrain_r{rank}_a{alpha}_ar{ratio_label}_{ATTN_RUN_LABEL}_e{EPOCHS}'
    run_root = OUTPUT_ROOT / 'runs' / run_name
    events = read_jsonl(run_root / 'training_events.jsonl')
    for ev in events:
        if ev.get('kind') == 'optimizer_step':
            step_rows.append({
                'run_name': run_name,
                'rank': rank,
                'alpha': alpha,
                'alpha_ratio': alpha_ratio,
                'epoch': ev.get('epoch'),
                'global_step': ev.get('global_step'),
                'micro_idx': ev.get('micro_idx'),
                'train_step_loss_lower_better': ev.get('loss'),
                'lr': ev.get('lr'),
            })
    hist_path = run_root / 'history.csv'
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
        last = hist.tail(1).iloc[0].to_dict()
    else:
        last = {}
    opt_path = run_root / 'optimizer_param_groups.json'
    cfg_path = run_root / 'model_trainable_config.json'
    trainable_params = math.nan
    if opt_path.exists():
        opt = json.loads(opt_path.read_text(encoding='utf-8'))
        trainable_params = sum(int(g.get('trainable_params', 0)) for g in opt.get('param_groups', []))
    attn = ''
    sdpa = ''
    if cfg_path.exists():
        cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
        attn = cfg.get('attn_implementation', '')
        sdpa = cfg.get('sdpa_kernel', '')
    summary_rows.append({
        'run_name': run_name,
        'rank': rank,
        'alpha': alpha,
        'alpha_ratio': alpha_ratio,
        'trainable_params': trainable_params,
        'attn_implementation': attn,
        'sdpa_kernel': sdpa,
        'global_step': last.get('global_step', math.nan),
        'train_loss_lower_better': last.get('train_loss', math.nan),
        'val_loss_lower_better': last.get('val_loss', math.nan),
        'val_token_accuracy_higher_better': last.get('val_token_accuracy_higher_better', math.nan),
        'epoch_elapsed_sec': last.get('epoch_elapsed_sec', math.nan),
    })

steps_df = pd.DataFrame(step_rows)
summary_df = pd.DataFrame(summary_rows)
plots_root = OUTPUT_ROOT / 'plots'
plots_root.mkdir(parents=True, exist_ok=True)
steps_csv = OUTPUT_ROOT / 'lora_rank_sweep_optimizer_steps.csv'
summary_csv = OUTPUT_ROOT / 'lora_rank_sweep_summary.csv'
steps_df.to_csv(steps_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

fig, ax = plt.subplots(figsize=(12, 7))
if not steps_df.empty:
    for (rank, alpha, alpha_ratio), group in steps_df.groupby(['rank', 'alpha', 'alpha_ratio'], sort=False):
        group = group.sort_values('global_step')
        label = f'train r={rank} a={alpha} ar={alpha_ratio}'
        line, = ax.plot(group['global_step'], group['train_step_loss_lower_better'], linewidth=1.5, alpha=0.85, label=label)
        srow = summary_df[(summary_df['rank'] == rank) & (summary_df['alpha'] == alpha) & (summary_df['alpha_ratio'] == alpha_ratio)]
        if not srow.empty:
            x = float(srow.iloc[0].get('global_step', group['global_step'].max()))
            y = srow.iloc[0].get('val_loss_lower_better')
            if pd.notna(y):
                ax.scatter([x], [float(y)], marker='x', s=90, linewidths=2.2, color=line.get_color(), label=f'val r={rank} a={alpha} ar={alpha_ratio}')
ax.set_title('One-Epoch Gemma SOKE LoRA Rank Sweep Loss (lower better)')
ax.set_xlabel('Optimizer step')
ax.set_ylabel('Loss (lower better)')
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plot_path = plots_root / 'lora_rank_sweep_train_val_loss.png'
fig.savefig(plot_path, dpi=180)
plt.show()

print('Wrote:', steps_csv)
print('Wrote:', summary_csv)
print('Wrote:', plot_path)
display(summary_df.sort_values('val_loss_lower_better'))

In [ ]:
# Sync sweep outputs back to Drive.
if SYNC_TO_DRIVE and DRIVE_BUNDLE_ROOT.exists():
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    run(['rsync', '-a', str(OUTPUT_ROOT) + '/', str(DRIVE_OUTPUT_ROOT) + '/'])
    print('Synced:', DRIVE_OUTPUT_ROOT)
else:
    print('Drive sync skipped.')

In [ ]:
# Optional Colab close/unassign-runtime cell. Disable with SOKE_GEMMA_SWEEP_UNASSIGN_RUNTIME=0.
if os.environ.get('SOKE_GEMMA_SWEEP_UNASSIGN_RUNTIME', '1') == '1':
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print('Runtime unassign skipped:', repr(exc))
else:
    print('Runtime unassign disabled by environment flag.')